# Composing a model from submodels

Every PyTES model is a fixed composition of submodels. `print_submodel_info` shows them: for
each slot, the class, the variables it requires and provides, and its assumptions. To try a
new submodel, build a `pytes.ComposedModel` from the submodels you want. The wiring is
checked as for the built-in models, and the result is an ordinary PyBaMM model.

In [1]:
import numpy as np
import pybamm

import pytes

base = pytes.SharpFrontModel()
base.print_submodel_info()

fluid: AdvectedFluid (Heat transfer fluid)
    requires:
        Heat transfer fluid heat gain per unit length [W.m-1]
        Initial uniform temperature [K]
    provides:
        Heat transfer fluid specific enthalpy [J.kg-1]
        Inlet specific enthalpy [J.kg-1]
        Outlet specific enthalpy [J.kg-1]
        Mass flow rate [kg.s-1]
        Heat transfer fluid thermal energy [J]
        Heat transfer fluid temperature [K]
        Heat transfer fluid temperature [degC]
        Inlet temperature [K]
        Inlet temperature [degC]
        Outlet temperature [K]
        Outlet temperature [degC]
    assumptions:
        The fluid is incompressible, with constant density.
        Axial conduction and dispersion in the fluid are neglected.
        The flow is the same in every flow path.
storage: EnthalpyStorage (Phase-change material storage)
    requires:
        Phase-change material boundary heat flux [W.m-2]
        Initial uniform temperature [K]
        Initial uniform liqui

Here the coupling's surface flux law is replaced by a nonlinear one. The submodels of the
base model are reused, not modified, and `defaults_from=base` takes its geometry, mesh, solver
and parameter values.

In [2]:
class CubicFlux(pytes.submodels.SurfaceFluxLaw):
    """Heat flux c (T_f - T_s)**3, positive into the storage"""

    def heat_flux(self, fluid, h_f, T_f, T_s):
        return pybamm.Parameter("Cubic flux coefficient [W.m-2.K-3]") * (T_f - T_s) ** 3


storage = base.submodels["storage"]
coupling = pytes.submodels.LedgerWallCoupling(
    base.param,
    base.param.fluid,
    storage.label,
    storage.storage_geometry,
    flux_law=CubicFlux(),
)
cubic = pytes.ComposedModel(
    {**base.submodels, "coupling": coupling},
    base.param,
    name="Cubic flux",
    defaults_from=base,
)
parameter_values = cubic.default_parameter_values
parameter_values["Cubic flux coefficient [W.m-2.K-3]"] = 40

Energy is still conserved exactly, whatever the flux law, because the coupling takes the
fluid's heat from the storage's own energy balance.

In [3]:
t = np.linspace(0, 3600, 121)
solutions = [
    pybamm.Simulation(base).solve([0, 3600], t_interp=t),
    pybamm.Simulation(cubic, parameter_values=parameter_values).solve(
        [0, 3600], t_interp=t
    ),
]
for solution in solutions:
    E = solution["Total thermal energy [J]"].entries
    discharged = solution["Discharged energy [J]"].entries
    print(
        solution.all_models[0].name,
        "- max conservation error [J]:",
        np.max(np.abs(E - E[0] + discharged)),
    )
pybamm.dynamic_plot(solutions, ["Outlet temperature [degC]", "Power [kW]"])

Sharp front model - max conservation error [J]: 6.938353180885315e-08
Cubic flux - max conservation error [J]: 7.264316082000732e-08


interactive(children=(FloatSlider(value=0.0, description='t', max=1.0, step=0.01), Output()), _dom_classes=('w…